In [ ]:
# knn_regression_preprocessed.py
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1) Load data
data = pd.read_csv("train.csv")  # adjust path if needed

# quick checks
print("Shape:", data.shape)
print(data.dtypes)
print("Missing values per column:\n", data.isna().sum())

# Replace 'price' below if your target column has a different name
TARGET = "price"
if TARGET not in data.columns:
    raise KeyError(f"Target column '{TARGET}' not found in dataset. Columns: {list(data.columns)}")

X = data.drop(TARGET, axis=1)
y = data[TARGET].values  # numeric expected for regression

# 2) Identify numeric and categorical columns
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
print("Numeric cols:", num_cols)
print("Categorical cols:", cat_cols)

# 3) Preprocessing pipelines
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    # In recent scikit-learn versions the parameter name changed from `sparse`
    # to `sparse_output`. Use sparse_output=False to get a dense array.
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ],
    remainder="drop",  # drop any other columns
)

# 4) Full pipeline: preprocessor -> KNN regressor
pipe = Pipeline([
    ("preproc", preprocessor),
    ("knn", KNeighborsRegressor()),
])

# 5) Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 6) Grid search (use neg_mean_squared_error scoring for regression)
param_grid = {
    "knn__n_neighbors": [3, 5, 7, 9, 11, 13],
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2],  # p=1 Manhattan, p=2 Euclidean
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring="neg_mean_squared_error", n_jobs=-1, return_train_score=True)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
best_cv_mse = -grid.best_score_
print(f"Best CV MSE: {best_cv_mse:.4f}, RMSE: {np.sqrt(best_cv_mse):.4f}")

# 7) Evaluate on test set
y_pred = grid.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Test RMSE: {rmse:.4f}")
print(f"Test MAE: {mae:.4f}")
print(f"Test R^2: {r2:.4f}")

# 8) Plot: k vs CV RMSE (extract from cv_results_)
results = pd.DataFrame(grid.cv_results_)
# mean_test_score stores NEG_MSE (because we used neg_mean_squared_error)
results["mean_test_rmse"] = np.sqrt(-results["mean_test_score"])

plt.figure(figsize=(8, 5))
for weight in results['param_knn__weights'].unique():
    subset = results[results['param_knn__weights'] == weight]
    # param_knn__n_neighbors is an object dtype, convert to int for plotting
    ks = subset["param_knn__n_neighbors"].astype(int)
    plt.plot(ks, subset["mean_test_rmse"], marker='o', label=f"weights={weight}")
plt.xlabel("Number of neighbors (k)")
plt.ylabel("CV RMSE")
plt.title("K vs CV RMSE")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# 9) Optional: numeric feature correlation heatmap
if len(num_cols) > 0:
    plt.figure(figsize=(8, 6))
    sns.heatmap(data[num_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f")
    plt.title("Numeric features correlation")
    plt.tight_layout()
    plt.show()
else:
    print("No numeric columns to show correlation heatmap.")


Shape: (75000, 4)
sample_id            int64
catalog_content     object
image_link          object
price              float64
dtype: object
Missing values per column:
 sample_id          0
catalog_content    0
image_link         0
price              0
dtype: int64
Numeric cols: ['sample_id']
Categorical cols: ['catalog_content', 'image_link']
